<a href="https://colab.research.google.com/github/ancestor9/2026_Fall_Learning-Langchain-AI-Agent/blob/main/rag-from-scratch/gemini_rag_from_scratch_1_to_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Rag From Scratch: Overview

These notebooks walk through the process of building RAG app(s) from scratch.

They will build towards a broader understanding of the RAG langscape, as shown here:



## Environment

`(1) Packages`

In [ ]:
! pip install langchain_community tiktoken langchain-google-genai langchainhub chromadb langchain -q

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.62.1 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
betterproto 2.0.0b7 requires python-dateutil<3.0,>=2.8, but you have python-dateutil 4.0.0 which is incompatible.
bigframes 2.51.0 requires python-dateutil<3,>=2.8.2, but you have python-dateutil 4.0.0 which is incompatible.
ibis-framework 9.5.0 requires python-dateutil<3,>=2.8.2, but you have python-dateutil 4.0.0 which is incompatible.
tweepy 4.17.0 requires oauthlib<4,>=3.2.0, but you have oauthlib 4.0.1 which is incompatible.
google-adk 2.7.1 requires google-genai<3,>=2.12.1, but you have google-genai 3.0.0 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have openteleme

`(2) LangSmith`

https://docs.smith.langchain.com/

In [ ]:
# import os
# os.environ['LANGCHAIN_TRACING_V2'] = 'true'
# os.environ['LANGCHAIN_ENDPOINT'] = 'https://api.smith.langchain.com'
# os.environ['LANGCHAIN_API_KEY'] = <your-api-key>

`(3) API Keys`

In [ ]:
import os
from google.colab import userdata
from langchain_google_genai import ChatGoogleGenerativeAI

YOUR_API_KEY = userdata.get('Gemini-API-Key')
os.environ["GOOGLE_API_KEY"] = YOUR_API_KEY

`(4) Embeddins`

In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

/tmp/ipykernel_1354/3531111585.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.embeddings import HuggingFaceEmbeddings
/tmp/ipykernel_1354/3531111585.py:2: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

... (4 outputs left out)

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

## Part 1: Overview

[RAG quickstart](https://python.langchain.com/docs/use_cases/question_answering/quickstart)

In [ ]:
import bs4
from langchain_core.prompts import ChatPromptTemplate
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import Chroma
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_google_genai import ChatGoogleGenerativeAI

#### INDEXING ####
loader = WebBaseLoader(
    web_paths=("https://lilianweng.github.io/posts/2023-06-23-agent/",),
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            class_=("post-content", "post-title", "post-header")
        )
    ),
)
docs = loader.load()

splits = RecursiveCharacterTextSplitter(
    chunk_size=1000, chunk_overlap=200
).split_documents(docs)

vectorstore = Chroma.from_documents(documents=splits, embedding=embeddings)
retriever = vectorstore.as_retriever() # <- 여기에 embeddings 객체가 포함되어 있음!

#### RETRIEVAL and GENERATION ####
# 지침 사항: "검색된 context를 바탕으로 질문에 답하고, 모르겠으면 모른다고 답할 것. 최대 3문장으로 간결하게 작성할 것"을 명시하고 있으며,
# {question}과 {context} 자리에 값이 동적으로 삽입
prompt = ChatPromptTemplate.from_template(
    "You are an assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer the question. "
    "If you don't know the answer, just say that you don't know. "
    "Use three sentences maximum and keep the answer concise.\n\n"
    "Question: {question}\n\nContext: {context}\n\nAnswer:"
)

llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash", temperature=0)

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# LangChain의 LCEL(LangChain Expression Language) 파이프 기능을 사용해 전체 RAG 흐름을 하나로 연결
rag_chain = (
    {
        "context": retriever | format_docs,  # 입력된 질문을 받아 관련 문서를 검색 및 포맷팅하고
        "question": RunnablePassthrough()    # 질문 자체는 그대로 전달

    }
    | prompt
    | llm
    | StrOutputParser()
)

rag_chain.invoke("What is Task Decomposition?")

'Task decomposition is a process that breaks down complex tasks into smaller, simpler, and more manageable steps. It can be achieved through methods such as LLM prompting (like Chain of Thought), task-specific instructions, or human inputs. This technique enables agents to plan ahead and parse user requests into actionable sub-tasks.'

## **코드 분석**

In [ ]:
# @title **1.  retriever는 내부적으로 사전에 등록해둔 embeddings 모델로 전달받은 질문 텍스트를 그 자리에서 수치 벡터(숫자 배열)로 변환(임베딩)**
retriever

VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x7ac0880e8e10>, search_kwargs={})

In [ ]:
# @title **2. Prompt : -> | RunnablePassthrough()를 추가해 LCEL 체인화**
runnable_map = {
    "context": retriever | format_docs,  # retriever는 내부적으로 사전에 등록해둔 embeddings 모델로 전달받은 질문 텍스트를 그 자리에서 수치 벡터(숫자 배열)로 변환(임베딩)
    "question": RunnablePassthrough()
} | RunnablePassthrough()

print(runnable_map)

# 정상 실행
inputs = map_chain.invoke("What is Task Decomposition?")
print(inputs)
print('\n')


first={
  context: VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x7ac0880e8e10>, search_kwargs={})
           | RunnableLambda(format_docs),
  question: RunnablePassthrough()
} middle=[] last=RunnablePassthrough()
{'context': 'Component One: Planning#\nA complicated task usually involves many steps. An agent needs to know what they are and plan ahead.\nTask Decomposition#\nChain of thought (CoT; Wei et al. 2022) has become a standard prompting technique for enhancing model performance on complex tasks. The model is instructed to “think step by step” to utilize more test-time computation to decompose hard tasks into smaller and simpler steps. CoT transforms big tasks into multiple manageable tasks and shed lights into an interpretation of the model’s thinking process.\nTree of Thoughts (Yao et al. 2023) extends CoT by exploring multiple reasoning possibilities at each step. It first decomposes the pr

In [ ]:
print("=== 1. 질문 (Question) ===")
print(inputs["question"])

print("\n=== 2. 검색된 문서 (Context) ===")
print(inputs["context"])

=== 1. 질문 (Question) ===
What is Task Decomposition?

=== 2. 검색된 문서 (Context) ===
Component One: Planning#
A complicated task usually involves many steps. An agent needs to know what they are and plan ahead.
Task Decomposition#
Chain of thought (CoT; Wei et al. 2022) has become a standard prompting technique for enhancing model performance on complex tasks. The model is instructed to “think step by step” to utilize more test-time computation to decompose hard tasks into smaller and simpler steps. CoT transforms big tasks into multiple manageable tasks and shed lights into an interpretation of the model’s thinking process.
Tree of Thoughts (Yao et al. 2023) extends CoT by exploring multiple reasoning possibilities at each step. It first decomposes the problem into multiple thought steps and generates multiple thoughts per step, creating a tree structure. The search process can be BFS (breadth-first search) or DFS (depth-first search) with each state evaluated by a classifier (via a prom

In [ ]:
def format_docs(docs):
    for i, doc in enumerate(docs):
        print(f"--- [문서 {i+1}] ---")
        print(f"📌 메타데이터: {doc.metadata}")
        print(f"📄 본문 (처음 200자): {doc.page_content[:200]}...\n")

    return "\n\n".join(doc.page_content for doc in docs)

In [ ]:
# 질문에 대해 검색된 문서 리스트 가져오기
retrieved_docs = retriever.invoke("What is Task Decomposition?")

# 검색된 모든 문서 정보 출력
for i, doc in enumerate(retrieved_docs):
    print(f"=== Document {i+1} ===")
    print(f"▶ Metadata: {doc.metadata}")
    print(f"▶ Page Content:\n{doc.page_content}\n")

=== Document 1 ===
▶ Metadata: {'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/'}
▶ Page Content:
Component One: Planning#
A complicated task usually involves many steps. An agent needs to know what they are and plan ahead.
Task Decomposition#
Chain of thought (CoT; Wei et al. 2022) has become a standard prompting technique for enhancing model performance on complex tasks. The model is instructed to “think step by step” to utilize more test-time computation to decompose hard tasks into smaller and simpler steps. CoT transforms big tasks into multiple manageable tasks and shed lights into an interpretation of the model’s thinking process.
Tree of Thoughts (Yao et al. 2023) extends CoT by exploring multiple reasoning possibilities at each step. It first decomposes the problem into multiple thought steps and generates multiple thoughts per step, creating a tree structure. The search process can be BFS (breadth-first search) or DFS (depth-first search) with each state evalu

In [ ]:
# @title **3. Vectorsore_ Chroma**
vectorstore

In [ ]:
# vectorstore의 데이터를 가져옵니다.
data = vectorstore.get()

# 저장된 전체 문서 개수 확인
print(f"총 저장된 문서 개수: {len(data['documents'])}")

# 처음 3개 문서의 내용과 메타데이터 출력
for i in range(min(3, len(data['documents']))):
    print(f"\n--- Document {i+1} ---")
    print(f"Metadata: {data['metadatas'][i]}")
    print(f"Content preview: {data['documents'][i][:300]}...")

총 저장된 문서 개수: 113

--- Document 1 ---
Metadata: {'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/'}
Content preview: LLM Powered Autonomous Agents
    
Date: June 23, 2023  |  Estimated Reading Time: 31 min  |  Author: Lilian Weng


Building agents with LLM (large language model) as its core controller is a cool concept. Several proof-of-concepts demos, such as AutoGPT, GPT-Engineer and BabyAGI, serve as inspiring...

--- Document 2 ---
Metadata: {'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/'}
Content preview: Memory

Short-term memory: I would consider all the in-context learning (See Prompt Engineering) as utilizing short-term memory of the model to learn.
Long-term memory: This provides the agent with the capability to retain and recall (infinite) information over extended periods, often by leveraging ...

--- Document 3 ---
Metadata: {'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/'}
Content preview: Component One: Planning#
A compli

## Part 2: Indexing

In [ ]:
# Documents
question = "What kinds of pets do I like?"
document = "My favorite pet is a cat."

[Count tokens](https://github.com/openai/openai-cookbook/blob/main/examples/How_to_count_tokens_with_tiktoken.ipynb) considering [~4 char / token](https://help.openai.com/en/articles/4936856-what-are-tokens-and-how-to-count-them)

In [ ]:
import tiktoken

def num_tokens_from_string(string: str, encoding_name: str) -> int:
    """Returns the number of tokens in a text string."""
    encoding = tiktoken.get_encoding(encoding_name)
    num_tokens = len(encoding.encode(string))
    return num_tokens

num_tokens_from_string(question, "cl100k_base")

8

[Text embedding models](https://python.langchain.com/docs/integrations/text_embedding/openai)

In [ ]:
embeddings

HuggingFaceEmbeddings(client=SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'BertModel'})
  (1): Pooling({'embedding_dimension': 384, 'pooling_mode': 'mean', 'include_prompt': True})
  (2): Normalize({})
), model_name='all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, multi_process=False, show_progress=False)

In [ ]:
query_result = embeddings.embed_query(question)
document_result = embeddings.embed_query(document)
len(query_result)

384

In [ ]:
query_result[:10]

[0.03080459125339985,
 -0.054369471967220306,
 0.10352050513029099,
 0.04905467852950096,
 -0.10118033736944199,
 0.01631731539964676,
 0.06456958502531052,
 -0.10090932250022888,
 0.03199333697557449,
 0.062347836792469025]

In [ ]:
document_result[:10]

[-0.007852793671190739,
 -0.018404575064778328,
 0.060350436717271805,
 0.014217151328921318,
 -0.09984561055898666,
 0.004160253331065178,
 0.08463754504919052,
 -0.021178418770432472,
 0.04404459521174431,
 0.05897689238190651]

[Cosine similarity](https://platform.openai.com/docs/guides/embeddings/frequently-asked-questions) is recommended (1 indicates identical) for OpenAI embeddings.

In [ ]:
import numpy as np

def cosine_similarity(vec1, vec2):
    dot_product = np.dot(vec1, vec2)
    norm_vec1 = np.linalg.norm(vec1)
    norm_vec2 = np.linalg.norm(vec2)
    return dot_product / (norm_vec1 * norm_vec2)

similarity = cosine_similarity(query_result, document_result)
print("Cosine Similarity:", similarity)

Cosine Similarity: 0.5782657349816056


[Document Loaders](https://python.langchain.com/docs/integrations/document_loaders/)

In [ ]:
#### INDEXING ####

# Load blog
import bs4
from langchain_community.document_loaders import WebBaseLoader
loader = WebBaseLoader(
    web_paths=("https://lilianweng.github.io/posts/2023-06-23-agent/",),
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            class_=("post-content", "post-title", "post-header")
        )
    ),
)
blog_docs = loader.load()

[Splitter](https://python.langchain.com/docs/modules/data_connection/document_transformers/recursive_text_splitter)

> This text splitter is the recommended one for generic text. It is parameterized by a list of characters. It tries to split on them in order until the chunks are small enough. The default list is ["\n\n", "\n", " ", ""]. This has the effect of trying to keep all paragraphs (and then sentences, and then words) together as long as possible, as those would generically seem to be the strongest semantically related pieces of text.

In [ ]:
# Split
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=300,
    chunk_overlap=50)

# Make splits
splits = text_splitter.split_documents(blog_docs)

[Vectorstores](https://python.langchain.com/docs/integrations/vectorstores/)

## Part 3: Retrieval

In [ ]:
# Index
from langchain_community.vectorstores import Chroma

vectorstore = Chroma.from_documents(documents=splits,
                                    embedding=embeddings)

retriever = vectorstore.as_retriever()

In [ ]:
retriever

VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x7ac0880e8e10>, search_kwargs={})

In [ ]:
docs = retriever.invoke("What is Task Decomposition?")

In [ ]:
len(docs)

4

In [ ]:
docs

[Document(metadata={'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/'}, page_content='Component One: Planning#\nA complicated task usually involves many steps. An agent needs to know what they are and plan ahead.\nTask Decomposition#\nChain of thought (CoT; Wei et al. 2022) has become a standard prompting technique for enhancing model performance on complex tasks. The model is instructed to “think step by step” to utilize more test-time computation to decompose hard tasks into smaller and simpler steps. CoT transforms big tasks into multiple manageable tasks and shed lights into an interpretation of the model’s thinking process.\nTree of Thoughts (Yao et al. 2023) extends CoT by exploring multiple reasoning possibilities at each step. It first decomposes the problem into multiple thought steps and generates multiple thoughts per step, creating a tree structure. The search process can be BFS (breadth-first search) or DFS (depth-first search) with each state evaluated by a

In [ ]:
for doc in docs:
    print(doc.page_content)
    print("\n")

Component One: Planning#
A complicated task usually involves many steps. An agent needs to know what they are and plan ahead.
Task Decomposition#
Chain of thought (CoT; Wei et al. 2022) has become a standard prompting technique for enhancing model performance on complex tasks. The model is instructed to “think step by step” to utilize more test-time computation to decompose hard tasks into smaller and simpler steps. CoT transforms big tasks into multiple manageable tasks and shed lights into an interpretation of the model’s thinking process.
Tree of Thoughts (Yao et al. 2023) extends CoT by exploring multiple reasoning possibilities at each step. It first decomposes the problem into multiple thought steps and generates multiple thoughts per step, creating a tree structure. The search process can be BFS (breadth-first search) or DFS (depth-first search) with each state evaluated by a classifier (via a prompt) or majority vote.
Task decomposition can be done (1) by LLM with simple prompt

## Part 4: Generation

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

# Prompt
template = """Answer the question based only on the following context:
{context}

Question: {question}
"""

prompt = ChatPromptTemplate.from_template(template)
prompt

ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='Answer the question based only on the following context:\n{context}\n\nQuestion: {question}\n'), additional_kwargs={})])

In [ ]:
# LLM
llm

ChatGoogleGenerativeAI(metadata={'lc_versions': {'langchain-core': '1.7.6', 'langchain': '1.4.8', 'langchain-google-genai': '4.4.3'}}, profile={'name': 'Gemini 3.5 Flash', 'release_date': '2026-05-19', 'last_updated': '2026-05-19', 'open_weights': False, 'max_input_tokens': 1048576, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': True, 'pdf_inputs': True, 'video_inputs': True, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'image_tool_message': True, 'tool_choice': True, 'reasoning_effort_levels': ['minimal', 'low', 'medium', 'high'], 'reasoning_effort_default': 'medium'}, google_api_key=SecretStr('**********'), model='gemini-3.5-flash', temperature=0.0, client=<google.genai.client.Client object at 0x7ac0604c2900>, default_metadata=(), model_kwargs={})

In [ ]:
# Chain
chain = prompt | llm

In [ ]:
# Run
chain.invoke({"context":docs,"question":"What is Task Decomposition?"})

AIMessage(content=[{'type': 'text', 'text': 'Based on the provided context, **Task Decomposition** is a planning process used to break down or decompose hard, complicated tasks into smaller, simpler, and more manageable steps or subgoals. \n\nIt can be accomplished through:\n1. **Chain of Thought (CoT):** Instructing a model to "think step by step" to utilize test-time computation to break tasks into simpler steps.\n2. **Tree of Thoughts:** Extending CoT by decomposing problems into multiple thought steps and exploring multiple reasoning possibilities at each step in a tree structure.\n3. **Simple LLM Prompting:** Using prompts such as *"Steps for XYZ.\\n1."* or *"What are the subgoals for achieving XYZ?"*\n4. **Task-Specific Instructions:** Providing targeted directions (e.g., asking for a *"story outline"* when writing a novel).\n5. **Human Inputs.**', 'extras': {'signature': 'EtMSCtASAWkUfROeCQ62qeX5l0afN31T2zZaxUPQukbLRSGvl7yYbCAeOazlg/P/92XGNv49Hmx86xm9bP9f6G58pj/fE3NlDNyX1LD4K9f0

In [ ]:
# from langchain import hub
# prompt_hub_rag = hub.pull("rlm/rag-prompt")

In [ ]:
# prompt_hub_rag

[RAG chains](https://python.langchain.com/docs/expression_language/get_started#rag-search-example)

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

rag_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

rag_chain.invoke("What is Task Decomposition?")

'Based on the provided context, **Task Decomposition** is the process of breaking down (or decomposing) complex or big tasks into smaller, simpler, and more manageable steps or subgoals. \n\nIt can be accomplished in a few ways:\n1. **By an LLM with simple prompting** (e.g., asking "Steps for XYZ.\\n1." or "What are the subgoals for achieving XYZ?").\n2. **By using task-specific instructions** (e.g., "Write a story outline." for writing a novel).\n3. **With human inputs**.\n\nTechniques like **Chain of Thought (CoT)** (instructing the model to "think step by step") and **Tree of Thoughts** (exploring multiple reasoning possibilities at each step) are used to perform task decomposition.'